# ✈️ LangGraph Travel Planner Flow

## What We're Building

A travel planning workflow that:
1. **Gathers preferences** — destination, budget, dates, interests
2. **Generates a plan** — LLM drafts a day-by-day itinerary
3. **Checkpoints** — saves state so the user can review and request revisions
4. **Revises the plan** — incorporates user feedback and regenerates

## Architecture

```
User Preferences
      │
      ▼
┌────────────┐    ┌────────────┐    ┌──────────┐    ┌────────────┐
│  Gather    │ ──▶│  Generate  │ ──▶│ ⏸ REVIEW │ ──▶│  Revise    │
│  Prefs     │    │  Plan      │    │ Checkpoint│    │  Plan      │
└────────────┘    └────────────┘    └──────────┘    └────────────┘
```

## Stack
- **LangGraph** — workflow with checkpointing for iterative revisions
- **Ollama** — local LLM for itinerary generation and revision

## Step 1 — Imports & Configuration

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3:4b", temperature=0.7)
print("All imports successful!")

## Step 2 — Define State

The state holds the traveler's preferences, the generated itinerary,
user feedback, and the revised version. The `revision_count` tracks
how many rounds of revision have occurred.

In [ ]:
class TravelState(TypedDict):
    destination: str
    budget_usd: int
    num_days: int
    interests: str
    travel_dates: str
    itinerary: str
    user_feedback: str
    revised_itinerary: str
    revision_count: int

print("TravelState defined")

## Step 3 — Define Graph Nodes

| Node | Role |
|------|------|
| `gather_preferences` | Validates and summarizes travel preferences |
| `generate_plan` | LLM creates a day-by-day itinerary |
| `revise_plan` | LLM revises the itinerary based on user feedback |

In [ ]:
def gather_preferences(state: TravelState) -> dict:
    """Validate and summarize travel preferences."""
    print("📋 Node: gather_preferences — Validating inputs...")
    dest = state["destination"].strip()
    if not dest:
        raise ValueError("Destination is required")
    days = state["num_days"]
    if days < 1 or days > 30:
        raise ValueError(f"num_days must be 1-30, got {days}")
    budget = state["budget_usd"]
    if budget < 100:
        raise ValueError(f"Budget too low: ${budget}")

    print(f"  ✓ Destination: {dest}")
    print(f"  ✓ Duration: {days} days")
    print(f"  ✓ Budget: ${budget:,}")
    print(f"  ✓ Interests: {state['interests']}")
    print(f"  ✓ Dates: {state['travel_dates']}")
    return {}


plan_prompt = ChatPromptTemplate.from_template(
    """You are an expert travel planner. Create a detailed day-by-day itinerary.

Destination: {destination}
Duration: {num_days} days
Budget: ${budget_usd} total
Travel Dates: {travel_dates}
Interests: {interests}

Create a day-by-day itinerary with:
- Morning, afternoon, and evening activities for each day
- Estimated cost per activity where relevant
- Restaurant or food recommendations
- Practical tips (transport, booking advice)
- A running budget tracker

Keep it realistic and budget-conscious."""
)


def generate_plan(state: TravelState) -> dict:
    """Generate the initial travel itinerary with LLM."""
    print("🗺️ Node: generate_plan — Creating itinerary...")
    chain = plan_prompt | llm | StrOutputParser()
    itinerary = chain.invoke({
        "destination": state["destination"],
        "num_days": state["num_days"],
        "budget_usd": state["budget_usd"],
        "travel_dates": state["travel_dates"],
        "interests": state["interests"],
    })
    print(f"     Generated {len(itinerary)} chars of itinerary")
    return {"itinerary": itinerary}


revise_prompt = ChatPromptTemplate.from_template(
    """You are an expert travel planner revising an itinerary based on feedback.

Original Itinerary:
{itinerary}

User Feedback:
{feedback}

Revise the itinerary to address the feedback while keeping the overall
structure and budget intact. Clearly mark what changed."""
)


def revise_plan(state: TravelState) -> dict:
    """Revise the itinerary based on user feedback."""
    count = state.get("revision_count", 0) + 1
    print(f"✏️ Node: revise_plan — Revision #{count}...")
    current = state.get("revised_itinerary") or state["itinerary"]
    chain = revise_prompt | llm | StrOutputParser()
    revised = chain.invoke({
        "itinerary": current,
        "feedback": state["user_feedback"],
    })
    print(f"     Revised itinerary: {len(revised)} chars")
    return {"revised_itinerary": revised, "revision_count": count}


print("All nodes defined")

## Step 4 — Build Graph with Checkpoint

We use `interrupt_before=["revise_plan"]` so the workflow pauses
after generating the plan, giving the user time to review and provide
feedback before the revision step runs.

In [ ]:
workflow = StateGraph(TravelState)

workflow.add_node("gather_preferences", gather_preferences)
workflow.add_node("generate_plan", generate_plan)
workflow.add_node("revise_plan", revise_plan)

workflow.set_entry_point("gather_preferences")
workflow.add_edge("gather_preferences", "generate_plan")
workflow.add_edge("generate_plan", "revise_plan")
workflow.add_edge("revise_plan", END)

memory = MemorySaver()
app = workflow.compile(
    checkpointer=memory,
    interrupt_before=["revise_plan"]  # Pause for user review
)

print("Travel planner workflow compiled (pauses before revision)")

## Step 5 — Run Phase 1: Gather & Generate

In [ ]:
config = {"configurable": {"thread_id": "travel-001"}}

result = app.invoke({
    "destination": "Tokyo, Japan",
    "budget_usd": 3000,
    "num_days": 5,
    "interests": "street food, temples, anime culture, nature walks",
    "travel_dates": "October 15-20, 2026",
    "itinerary": "",
    "user_feedback": "",
    "revised_itinerary": "",
    "revision_count": 0,
}, config)

print("\n" + "="*60)
print("🗺️ GENERATED ITINERARY (review before revising)")
print("="*60)
print(result["itinerary"])
print("\n⏸️ Workflow paused — provide feedback to revise...")

## Step 6 — User Provides Feedback & Revision Runs

The user reviews the plan, then provides feedback.
We inject the feedback into state and resume the graph.

In [ ]:
user_feedback = """
I'd like to:
- Swap Day 2 afternoon for a visit to Akihabara (anime district)
- Add a day trip to Mount Takao on Day 4 instead of city activities
- Include a ramen tour recommendation for at least one evening
- Keep the budget under $3000
"""

app.update_state(config, {"user_feedback": user_feedback})

print("🔄 Resuming with user feedback...\n")
final = app.invoke(None, config)

print("\n" + "="*60)
print("✏️ REVISED ITINERARY")
print("="*60)
print(final["revised_itinerary"])
print(f"\nRevision count: {final['revision_count']}")

## Step 7 — Inspect Checkpoint History

LangGraph's `MemorySaver` keeps every state snapshot.
This is useful for auditing how the plan evolved.

In [ ]:
print("📜 Checkpoint History")
print("="*40)
for i, snapshot in enumerate(app.get_state_history(config)):
    ts = snapshot.config.get("configurable", {}).get("thread_ts", "?")
    node = snapshot.metadata.get("source", "unknown") if snapshot.metadata else "unknown"
    has_itinerary = bool(snapshot.values.get("itinerary", ""))
    has_revised = bool(snapshot.values.get("revised_itinerary", ""))
    print(f"  Snapshot {i}: source={node}, itinerary={'✓' if has_itinerary else '✗'}, revised={'✓' if has_revised else '✗'}, ts={ts}")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Preference gathering** | Validates inputs before sending to LLM |
| **LLM generation** | Creates a structured day-by-day itinerary |
| **Checkpointing** | `MemorySaver` + `interrupt_before` pauses for review |
| **State injection** | `update_state` injects user feedback mid-flow |
| **Iterative revision** | Graph resumes and LLM revises based on feedback |

## Why Checkpoints for Travel Planning?

- Plans are subjective — users need to review before finalizing
- Checkpointing preserves the full revision history
- Easy to add multiple revision rounds without rebuilding
- State history shows how the plan evolved

## 🔧 Extensions

- **Multi-round revisions**: Loop `revise_plan` back with a conditional edge
- **Budget validator node**: Check that suggested activities stay within budget
- **External APIs**: Integrate flight/hotel price lookups
- **Export**: Generate a PDF or calendar file from the final itinerary